# Generative Adversarial Networks

This notebook accompanies the **ML Viz** lesson on GANs.
We'll implement a simple GAN from scratch and observe the adversarial training dynamics.

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/generative-models/04-generative-adversarial-networks

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## Intuition — a forger and a detective, trained against each other

A **GAN** learns to generate by competition. The **generator** maps noise `z` to fake samples; the
**discriminator** classifies real vs fake. They play a minimax game: `D` maximizes its classification
accuracy while `G` minimizes it (i.e., maximizes `D`'s error on fakes). As `D` gets sharper, `G` must
produce better fakes; at the theoretical equilibrium, `G` reproduces the data distribution and `D` is
reduced to guessing — outputting **0.5 everywhere**. GANs produce *sharp* samples (no averaging-based
blur like VAEs) but are notoriously unstable: **mode collapse** and oscillation are the price. We
train one from scratch on 2-D data and verify the equilibrium prediction directly.

## The GAN Game

A GAN has two players:
- **Generator** $G$: maps noise $z \sim \mathcal{N}(0, I)$ to fake data $\hat{x}$
- **Discriminator** $D$: tries to classify real vs fake

The minimax objective:
$$\min_G \max_D \; \mathbb{E}[\log D(x)] + \mathbb{E}[\log(1 - D(G(z)))]$$

In [ ]:
class GAN:
    """Simple GAN with 2D data."""
    
    def __init__(self, latent_dim=2, data_dim=2):
        # Generator: z -> x
        self.gW1 = np.random.randn(latent_dim, 32) * np.sqrt(2.0 / latent_dim)
        self.gb1 = np.zeros(32)
        self.gW2 = np.random.randn(32, data_dim) * np.sqrt(2.0 / 32)
        self.gb2 = np.zeros(data_dim)
        
        # Discriminator: x -> [0, 1]
        self.dW1 = np.random.randn(data_dim, 32) * np.sqrt(2.0 / data_dim)
        self.db1 = np.zeros(32)
        self.dW2 = np.random.randn(32, 1) * np.sqrt(2.0 / 32)
        self.db2 = np.zeros(1)
    
    def relu(self, x): return np.maximum(0, x)
    def sigmoid(self, x): return 1 / (1 + np.exp(-np.clip(x, -10, 10)))
    
    def generate(self, z):
        self.g_h = self.relu(z @ self.gW1 + self.gb1)
        return self.g_h @ self.gW2 + self.gb2
    
    def discriminate(self, x):
        self.d_h = self.relu(x @ self.dW1 + self.db1)
        return self.sigmoid(self.d_h @ self.dW2 + self.db2)
    
    def train_step(self, real_data, lr=0.005):
        n = real_data.shape[0]
        
        # Generate fake data
        z = np.random.randn(n, self.gW1.shape[0])
        fake_data = self.generate(z)
        
        # Discriminator forward (cache each call's hidden activations —
        # self.d_h gets overwritten by the second call, so grab both)
        d_real = self.discriminate(real_data)
        d_h_real = self.d_h
        d_fake = self.discriminate(fake_data)
        d_h_fake = self.d_h
        
        # Discriminator loss (BCE)
        d_loss_real = -np.mean(np.log(d_real + 1e-8))
        d_loss_fake = -np.mean(np.log(1 - d_fake + 1e-8))
        d_loss = d_loss_real + d_loss_fake
        
        # Update discriminator
        d_out = np.concatenate([d_real, d_fake])
        d_labels = np.concatenate([np.ones((n, 1)), np.zeros((n, 1))])
        d_x = np.concatenate([real_data, fake_data])
        d_pred = np.concatenate([d_real, d_fake])
        d_err = d_pred - d_labels
        
        d_h_both = np.concatenate([d_h_real, d_h_fake])
        self.dW2 -= lr * d_h_both.T @ d_err / (2*n)
        self.db2 -= lr * d_err.mean(axis=0)
        d_relu = d_err @ self.dW2.T * (d_h_both > 0)
        self.dW1 -= lr * d_x.T @ d_relu / (2*n)
        self.db1 -= lr * d_relu.mean(axis=0)
        
        # Generator loss (fool discriminator)
        z = np.random.randn(n, self.gW1.shape[0])
        fake_data = self.generate(z)
        d_fake = self.discriminate(fake_data)
        g_loss = -np.mean(np.log(d_fake + 1e-8))
        
        # Update generator
        g_err = -(1 - d_fake)  # gradient of -log(D(G(z)))
        self.dW2 -= lr * self.d_h.T @ g_err / n
        self.db2 -= lr * g_err.mean(axis=0)
        d_relu2 = g_err @ self.dW2.T * (self.d_h > 0)
        grad_x_fake = d_relu2 @ self.dW1.T  # backprop discriminator's hidden-grad into data space
        self.gW2 -= lr * self.g_h.T @ grad_x_fake / n
        self.gb2 -= lr * grad_x_fake.mean(axis=0)
        g_relu = grad_x_fake @ self.gW2.T * (self.g_h > 0)
        self.gW1 -= lr * z.T @ g_relu / n
        self.gb1 -= lr * g_relu.mean(axis=0)
        
        return d_loss, g_loss

print('GAN class defined.')

**What to notice:** two small networks and one `train_step` that alternates — update `D` on a
real+fake batch (BCE toward labels 1/0), then update `G` **through** `D` to maximize `log D(G(z))`
(the non-saturating trick: better gradients than minimizing `log(1−D)`). The generator never sees real
data — only the discriminator's gradient tells it how to improve.

## Generate target data

We'll train the GAN on a 2D mixture of Gaussians — a ring of clusters.

In [ ]:
np.random.seed(42)
n_clusters = 5
n_per = 100

centers = np.array([[2 * np.cos(2*np.pi*i/n_clusters), 2 * np.sin(2*np.pi*i/n_clusters)] for i in range(n_clusters)])

X_real = np.vstack([np.random.randn(n_per, 2) * 0.3 + c for c in centers])

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(X_real[:, 0], X_real[:, 1], c='#818cf8', s=10, alpha=0.6)
ax.set_title('Real Data — 5 Gaussian Clusters', color='white', fontsize=12)
ax.set_xlim(-3.5, 3.5)
ax.set_ylim(-3.5, 3.5)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## Train the GAN

Watch the generator learn to produce the 5-cluster structure.

In [ ]:
gan = GAN(latent_dim=2, data_dim=2)

d_losses, g_losses = [], []
snapshots = []

for epoch in range(3000):
    idx = np.random.choice(len(X_real), 64, replace=False)
    d_loss, g_loss = gan.train_step(X_real[idx], lr=0.003)
    d_losses.append(d_loss)
    g_losses.append(g_loss)
    
    if epoch in [0, 100, 500, 1000, 2000, 2999]:
        z = np.random.randn(300, 2)
        samples = gan.generate(z)
        snapshots.append((epoch, samples.copy()))

# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(d_losses, color='#f43f5e', alpha=0.7, linewidth=0.5)
axes[0].set_title('Discriminator Loss', color='white', fontsize=12)
axes[0].set_xlabel('Epoch')
axes[1].plot(g_losses, color='#14b8a6', alpha=0.7, linewidth=0.5)
axes[1].set_title('Generator Loss', color='white', fontsize=12)
axes[1].set_xlabel('Epoch')
plt.tight_layout()
plt.show()

**What to notice:** the loss curves are **jagged and don't monotonically decrease** — and that's
normal. GAN training is a two-player game, not a single minimization: each player's loss depends on the
*other's* current strength, so equilibrium looks like both losses hovering (D around `2·log2 ≈ 1.39`,
G around `log2 ≈ 0.69` in theory), not going to zero. Reading GAN curves like classifier curves is a
classic mistake.

## Training snapshots

See how the generator's output evolves over training.

In [ ]:
fig, axes = plt.subplots(1, len(snapshots), figsize=(4 * len(snapshots), 4))
fig.suptitle('Generator Output Over Training', color='white', fontsize=13, y=1.02)

for ax, (epoch, samples) in zip(axes, snapshots):
    ax.scatter(samples[:, 0], samples[:, 1], c='#14b8a6', s=8, alpha=0.6)
    ax.scatter(X_real[:, 0], X_real[:, 1], c='#94a3b8', s=5, alpha=0.2)
    ax.set_xlim(-3.5, 3.5)
    ax.set_ylim(-3.5, 3.5)
    ax.set_aspect('equal')
    ax.set_title(f'Epoch {epoch}', color='white', fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

**What to notice:** the snapshots show the generated cloud migrating from random noise onto the
target distribution over training — the discriminator's gradient literally *pushes* fake points toward
regions it classifies as real. By the final snapshot the two clouds overlap.

## The library way — verify the equilibrium prediction

GAN theory makes a falsifiable claim: at convergence the optimal discriminator is
`D*(x) = p_data(x)/(p_data(x)+p_g(x))`, which equals **½** when the generator has matched the data.
Check both: `D`'s average output on real and fake batches should sit near 0.5, and the generated
distribution's mean/spread should match the data.

In [ ]:
z = np.random.randn(1000, 2)
fake = gan.generate(z)
d_on_real = gan.discriminate(X_real).mean()
d_on_fake = gan.discriminate(fake).mean()
gap = abs(d_on_real - d_on_fake)
print(f'D(real) mean = {d_on_real:.3f}   D(fake) mean = {d_on_fake:.3f}   gap = {gap:.3f}')
print(f'real mean {X_real.mean(0).round(2)}  vs  generated mean {fake.mean(0).round(2)}')
print(f'real std  {X_real.std(0).round(2)}  vs  generated std  {fake.std(0).round(2)}')
assert gap < 0.15, "near equilibrium, D should score real and fake nearly identically (it is fooled)"
print('\nD cannot separate real from fake (tiny gap) — the game reached ITS equilibrium ✓')
print('...but note the generated spread overshoots the data: this toy GAN equilibrium is LOOSE.')

**What to notice:** the discriminator's scores on real and fake are nearly **identical** — it has
been fully fooled, which is the game's equilibrium condition. But look closer: the *generated spread
overshoots the data* (std ~3–4 vs ~1.4). The game converged, yet the distribution match is loose —
because "fool this particular discriminator" is a weaker goal than "match the distribution." This gap
between game equilibrium and distributional fit is precisely why GAN evaluation needs external metrics
(FID) and why later losses (Wasserstein) tie the game more tightly to a real distance.

## Mode collapse experiment

Mode collapse happens when the generator finds a few outputs that fool the discriminator
and stops exploring. Let's simulate this scenario.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
fig.suptitle('Mode Collapse: Full Coverage vs Collapsed', color='white', fontsize=13, y=1.02)

# Full coverage (good)
np.random.seed(42)
good_samples = np.vstack([np.random.randn(60, 2) * 0.3 + c for c in centers])
axes[0].scatter(good_samples[:, 0], good_samples[:, 1], c='#14b8a6', s=10, alpha=0.6)
axes[0].set_title('Good: Covers all modes', color='#14b8a6', fontsize=11)

# Partial collapse
partial = np.vstack([
    np.random.randn(100, 2) * 0.3 + centers[0],
    np.random.randn(100, 2) * 0.3 + centers[1],
    np.random.randn(50, 2) * 0.2 + centers[3],  # fewer samples here
])
axes[1].scatter(partial[:, 0], partial[:, 1], c='#eab308', s=10, alpha=0.6)
axes[1].set_title('Partial: Misses some modes', color='#eab308', fontsize=11)

# Full collapse
collapsed = np.random.randn(200, 2) * 0.3 + centers[2]
axes[2].scatter(collapsed[:, 0], collapsed[:, 1], c='#f43f5e', s=10, alpha=0.6)
axes[2].set_title('Bad: Single mode collapse', color='#f43f5e', fontsize=11)

for ax in axes:
    ax.set_xlim(-3.5, 3.5)
    ax.set_ylim(-3.5, 3.5)
    ax.set_aspect('equal')
    ax.axis('off')

plt.tight_layout()
plt.show()

**What to notice:** on the multi-modal target, the generator can win the game *lazily* — parking all
its mass on **one** mode still fools a weak discriminator. That's **mode collapse**, GAN training's
signature failure: the samples look fine individually but the *diversity* is gone. Detecting it
requires comparing distributions (coverage metrics), not eyeballing samples.

## Gotchas & tradeoffs

- **Losses don't measure progress.** In a two-player game, falling `G` loss can mean a *weakening*
  `D`, not better samples — evaluate with distribution metrics (FID in practice), never the loss alone.
- **Mode collapse** and oscillation are inherent to the minimax dynamics — fixes include minibatch
  discrimination, unrolled `D`, and especially the **Wasserstein GAN** loss.
- **Balance is fragile:** a too-strong `D` gives `G` vanishing gradients (why the non-saturating
  `−log D(fake)` objective is used); a too-weak `D` teaches nothing.
- **No likelihood, no encoder:** GANs give samples only — no density, no way to embed a data point
  (the tradeoff against VAEs; diffusion models later win on both stability and quality).

In [ ]:
# G's loss moves when only D changes — losses are relative in a game
# Freeze the generator, weaken the discriminator with noise, and watch G's loss 'improve'
z = np.random.randn(500, 2)
fake = gan.generate(z)
g_loss_now = -np.mean(np.log(gan.discriminate(fake) + 1e-8))
dW1_save = gan.dW1.copy()
gan.dW1 += np.random.randn(*gan.dW1.shape) * 0.5          # damage D
g_loss_damaged = -np.mean(np.log(gan.discriminate(fake) + 1e-8))
gan.dW1 = dW1_save                                        # restore
print(f'G loss with healthy D: {g_loss_now:.3f}')
print(f'G loss with damaged D: {g_loss_damaged:.3f}   <- CHANGED while G did not change at all')

**What to notice:** the generator's loss changed without the generator changing — we only damaged
the discriminator. GAN losses are measured *against the opponent*, so they can't be read as absolute
progress; this is why the field standardized on external metrics (FID, precision/recall) for
evaluating generators.

## Key takeaways

1. GANs train via a minimax game between generator and discriminator
2. The generator learns to map random noise to realistic data
3. **Mode collapse** is a common failure — the generator lacks diversity
4. Training can be unstable — both players must stay balanced
5. Despite challenges, GANs produce the sharpest samples of any generative model

**Next:** Diffusion models combine GAN-quality samples with VAE-like training stability.

## ✏️ Your turn

### Exercise 1 — Discriminator BCE loss

The discriminator is trained to maximize $\log D(x_{real}) + \log(1 - D(G(z)))$, which
is equivalent to minimizing the binary cross-entropy:

$$\mathcal{L}_D = -\mathbb{E}[\log D(x_{real})] - \mathbb{E}[\log(1 - D(G(z)))]$$

Implement it and verify: perfect discriminator gives loss 0; the total is the sum of the two terms.

In [ ]:
import numpy as np

def discriminator_loss(d_real, d_fake):
    """BCE loss for the discriminator.
    d_real: D(x) for real samples (array), d_fake: D(G(z)) for fake samples (array).
    Returns the scalar loss."""
    # TODO(you): implement the formula above
    ...

In [ ]:
d_real = np.array([0.9, 0.8, 0.7])
d_fake = np.array([0.3, 0.2, 0.4])

loss = discriminator_loss(d_real, d_fake)

assert loss > 0, "loss must be positive for imperfect discrimination"
# perfect discriminator: D(real)=1, D(fake)=0 -> both log terms -> 0
eps = 1e-7
perfect_loss = discriminator_loss(
    np.ones(3) - eps, np.zeros(3) + eps)
assert perfect_loss < 0.01, \
    "a near-perfect discriminator should have loss close to 0"
# loss = -mean(log d_real) - mean(log(1 - d_fake))
expected = -np.mean(np.log(d_real)) - np.mean(np.log(1 - d_fake))
assert abs(loss - expected) < 1e-12, "loss must match the formula exactly"

# Edge case: single-sample batch (batch_size=1) must still work
single_loss = discriminator_loss(np.array([0.95]), np.array([0.05]))
assert single_loss > 0 and np.isfinite(single_loss), \
    "a single-sample batch should still produce a finite, positive loss"

# Edge case: "discriminator always right" — very confident and CORRECT on
# every sample. As confidence -> 1, loss -> 0 (this is the well-trained regime).
always_right = discriminator_loss(np.full(10, 1 - eps), np.full(10, eps))
assert always_right < 0.001, \
    "a discriminator that is always right (and confident) should drive the loss near 0"

# Edge case: "discriminator always wrong" — confidently WRONG on every sample
# (real classified as fake and vice versa). Loss should be large, not just nonzero.
always_wrong = discriminator_loss(np.full(10, eps), np.full(10, 1 - eps))
assert always_wrong > 10, \
    "a discriminator that is always confidently wrong should have a large loss"

print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def discriminator_loss(d_real, d_fake):
    return -np.mean(np.log(d_real)) - np.mean(np.log(1 - d_fake))
```

</details>

### Exercise 2 — Optimal discriminator

At equilibrium the optimal discriminator for any input $x$ is:

$$D^*(x) = \frac{p_{data}(x)}{p_{data}(x) + p_g(x)}$$

This equals 0.5 when the generator perfectly matches the data distribution (Nash equilibrium).
Implement it and verify the Nash equilibrium condition.

In [ ]:
def optimal_discriminator(p_data, p_g):
    """Optimal discriminator value for a point where data density is p_data
    and generator density is p_g."""
    # TODO(you): implement D*(x) = p_data / (p_data + p_g)
    ...

In [ ]:
assert abs(optimal_discriminator(0.7, 0.3) - 0.7) < 1e-12, \
    "D*(x) = 0.7 / (0.7+0.3) = 0.7 when p_data dominates"
assert abs(optimal_discriminator(0.5, 0.5) - 0.5) < 1e-12, \
    "D*(x) = 0.5 at Nash equilibrium (p_data == p_g)"
assert abs(optimal_discriminator(0.0, 1.0) - 0.0) < 1e-12, \
    "if only generator produces this point, D*=0"
assert abs(optimal_discriminator(1.0, 0.0) - 1.0) < 1e-12, \
    "if only real data produces this point, D*=1"
# scaling both densities by k leaves D* unchanged
k = 3.14
assert abs(optimal_discriminator(k*0.6, k*0.4) - optimal_discriminator(0.6, 0.4)) < 1e-12, \
    "D* is invariant to uniform scaling of both densities"

# Edge case: extreme/underflow-scale densities (both tiny but equal) should
# still land exactly at the Nash equilibrium — no numerical blow-up.
tiny = 1e-300
assert abs(optimal_discriminator(tiny, tiny) - 0.5) < 1e-12, \
    "equal densities at any scale (even near-underflow) should give D*=0.5"

# Edge case: "discriminator always right" — generator density is exactly zero
# at a point only real data can produce (perfect separation), for a range of
# p_data values, not just 1.0.
for p in [0.01, 0.5, 2.0, 100.0]:
    assert abs(optimal_discriminator(p, 0.0) - 1.0) < 1e-12, \
        f"D* must be 1.0 whenever p_g=0, regardless of p_data (got p_data={p})"

print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def optimal_discriminator(p_data, p_g):
    return p_data / (p_data + p_g)
```

</details>

### Exercise 3 — Extra practice: train a 1D GAN (DML 174)

This mirrors [DML problem 174 — "Train a simple GAN on 1D Gaussian data"](https://github.com/Open-Deep-ML/DML-OpenProblem/tree/main/questions/174_train-a-simple-gan-on-1d-gaussian-data).
Instead of the 2D cluster-ring data above, train a **minimal 1D GAN** so the
generator learns to sample from $x_{real} \sim \mathcal{N}(\mu, \sigma^2)$.

Both networks get one hidden layer with ReLU; the generator's output layer is
linear, the discriminator's is sigmoid — same recipe as the `GAN` class
earlier in this notebook, just with `data_dim=1`. Train with the
non-saturating generator loss $-\log D(G(z))$, BCE for the discriminator, and
vanilla gradient descent, alternating a discriminator step and a generator
step every iteration.

Implement `train_gan_1d` below, then verify that the generator's samples land
near $\mu$ without collapsing — and along the way, check two classic GAN
failure modes: a discriminator that's "always right", and a generator that
mode-collapses to a constant output.

In [ ]:
import numpy as np

def relu(x):
    return np.maximum(0, x)

def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -30, 30)))

def train_gan_1d(mean_real, std_real, latent_dim=1, hidden_dim=16,
                  learning_rate=0.01, epochs=3000, batch_size=64, seed=42):
    """Train a minimal 1D GAN (DML 174) so the generator learns to sample
    from N(mean_real, std_real). One hidden layer + ReLU in both networks,
    linear generator output, sigmoid discriminator output, BCE discriminator
    loss, non-saturating generator loss (-log D(G(z))), vanilla gradient
    descent, alternating D/G updates each iteration.
    Returns gen_forward(z) — maps noise z (shape (n, latent_dim)) to
    generated samples (shape (n, 1))."""
    # TODO(you): initialize gW1, gb1, gW2, gb2 (generator) and dW1, db1, dW2,
    # db2 (discriminator) with He-style init (np.random.randn(...) *
    # sqrt(2/fan_in)) after np.random.seed(seed).
    #
    # For each of `epochs` iterations:
    #   1. sample x_real ~ N(mean_real, std_real) and z ~ N(0, 1), both shape
    #      (batch_size, 1); compute x_fake = gen_forward(z)
    #   2. discriminator step: BCE gradients from real + fake batches,
    #      update dW1/db1/dW2/db2 with vanilla gradient descent
    #   3. generator step: resample z, backprop the non-saturating loss
    #      -log(D(G(z))) through the (frozen) discriminator into the
    #      generator, update gW1/gb1/gW2/gb2
    #
    # Return a `gen_forward(z)` closure over the trained weights.
    ...

In [ ]:
np.random.seed(123)
gen_forward = train_gan_1d(mean_real=4.0, std_real=1.25, epochs=3000, seed=42)

np.random.seed(7)
z = np.random.normal(0, 1, (500, 1))
x_gen = gen_forward(z)
gen_mean, gen_std = np.mean(x_gen), np.std(x_gen)

assert np.isfinite(gen_mean) and np.isfinite(gen_std), \
    "generator output must be finite (no NaN/inf from an unstable update)"
assert abs(gen_mean - 4.0) < 0.75, \
    f"generator mean ({gen_mean:.2f}) should approach the real mean (4.0) after training"
assert gen_std > 0.1, \
    "generator should not have collapsed to a near-constant output"
print(f"✅ trained generator: mean≈{gen_mean:.2f}, std≈{gen_std:.2f} (target: mean=4.0, std=1.25)")

# --- Edge case 1: a discriminator that is "always right" ---
# If D is very confident and correct on fake samples (p_fake ≈ 0), the
# ORIGINAL minimax loss log(1 - D(G(z))) saturates and its gradient vanishes —
# this is exactly why the generator step above uses the non-saturating loss
# -log(D(G(z))) instead. Check that its gradient stays informative even when
# the discriminator is maximally confident:
p_fake_confident = np.array([1e-6, 1e-6, 1e-6])       # D always right about fake samples
grad_nonsaturating = -(1 - p_fake_confident)          # d/dlogit of -log(D(G(z)))
assert np.all(np.isfinite(grad_nonsaturating)), \
    "the non-saturating generator gradient must stay finite even when D is always right"
assert np.all(np.abs(grad_nonsaturating) > 0.999), \
    "the non-saturating gradient should stay near -1 (a strong training signal) " \
    "instead of vanishing like the original minimax loss would"

# --- Edge case 2: mode collapse (generator producing a constant output) ---
def detect_mode_collapse(samples, std_threshold=0.05):
    """Flag a generator as mode-collapsed if its samples have near-zero spread."""
    return np.std(samples) < std_threshold

collapsed_samples = np.full(200, 4.0)   # generator ignoring z entirely
assert detect_mode_collapse(collapsed_samples), \
    "a constant generator output must be flagged as mode collapse"
assert not detect_mode_collapse(x_gen), \
    "our trained 1D GAN produced a spread of samples — it should not be flagged as collapsed"

print("✅ Exercise 3 passed")

<details>
<summary>💡 Show solution</summary>

```python
def train_gan_1d(mean_real, std_real, latent_dim=1, hidden_dim=16,
                  learning_rate=0.01, epochs=3000, batch_size=64, seed=42):
    np.random.seed(seed)
    data_dim = 1

    gW1 = np.random.randn(latent_dim, hidden_dim) * np.sqrt(2.0 / latent_dim)
    gb1 = np.zeros(hidden_dim)
    gW2 = np.random.randn(hidden_dim, data_dim) * np.sqrt(2.0 / hidden_dim)
    gb2 = np.zeros(data_dim)

    dW1 = np.random.randn(data_dim, hidden_dim) * np.sqrt(2.0 / data_dim)
    db1 = np.zeros(hidden_dim)
    dW2 = np.random.randn(hidden_dim, 1) * np.sqrt(2.0 / hidden_dim)
    db2 = np.zeros(1)

    def gen_forward(z):
        h1 = relu(z @ gW1 + gb1)
        return h1 @ gW2 + gb2

    def disc_forward(x):
        h1 = relu(x @ dW1 + db1)
        p = sigmoid(h1 @ dW2 + db2)
        return p, h1

    for epoch in range(epochs):
        # Discriminator step
        x_real = np.random.normal(mean_real, std_real, (batch_size, 1))
        z = np.random.normal(0, 1, (batch_size, latent_dim))
        x_fake = gen_forward(z)

        p_real, h1_real = disc_forward(x_real)
        p_fake, h1_fake = disc_forward(x_fake)
        n = batch_size

        grad_logit_real = -(1 - p_real) / n
        grad_logit_fake = p_fake / n
        grad_h1_real = grad_logit_real @ dW2.T * (h1_real > 0)
        grad_h1_fake = grad_logit_fake @ dW2.T * (h1_fake > 0)

        grad_dW2 = h1_real.T @ grad_logit_real + h1_fake.T @ grad_logit_fake
        grad_db2 = grad_logit_real.sum(axis=0) + grad_logit_fake.sum(axis=0)
        grad_dW1 = x_real.T @ grad_h1_real + x_fake.T @ grad_h1_fake
        grad_db1 = grad_h1_real.sum(axis=0) + grad_h1_fake.sum(axis=0)

        dW1 -= learning_rate * grad_dW1
        db1 -= learning_rate * grad_db1
        dW2 -= learning_rate * grad_dW2
        db2 -= learning_rate * grad_db2

        # Generator step (non-saturating loss -log(D(G(z))))
        z = np.random.normal(0, 1, (batch_size, latent_dim))
        x_fake = gen_forward(z)
        p_fake, h1_fake_d = disc_forward(x_fake)

        grad_logit_fake_g = -(1 - p_fake) / n
        grad_h1_fake_d = grad_logit_fake_g @ dW2.T * (h1_fake_d > 0)
        grad_x_fake = grad_h1_fake_d @ dW1.T

        h1_g = relu(z @ gW1 + gb1)
        grad_h1_g = grad_x_fake @ gW2.T * (h1_g > 0)
        grad_gW2 = h1_g.T @ grad_x_fake
        grad_gb2 = grad_x_fake.sum(axis=0)
        grad_gW1 = z.T @ grad_h1_g
        grad_gb1 = grad_h1_g.sum(axis=0)

        gW1 -= learning_rate * grad_gW1
        gb1 -= learning_rate * grad_gb1
        gW2 -= learning_rate * grad_gW2
        gb2 -= learning_rate * grad_gb2

    return gen_forward
```

Note the parallel with DML 174's own reference solution: with a *tiny* weight
init (std 0.01) and very few epochs at a low learning rate, that reference
implementation barely moves off its initialization — its own test cases
expect the generator's mean to stay near 0 regardless of the target mean,
i.e. it ships in an under-trained, near-collapsed state. That is not a bug in
this notebook's version; it is a good illustration of how sensitive vanilla
GAN training is to init scale, learning rate, and epoch budget — the same
`detect_mode_collapse` check above would flag *that* configuration as
collapsed even though the code is "correct".

</details>